In [3]:
import pandas as pd
import numpy as np

# 1. Load the data (make sure we are loading the CSV we just saved!)
full_df = pd.read_csv('../data/dataset/preprocessed_full_data.csv')

# 2. Create cyclical time features
full_df['hour_sin'] = np.sin(2 * np.pi * full_df['Hour'] / 23.0)
full_df['hour_cos'] = np.cos(2 * np.pi * full_df['Hour'] / 23.0)

# 3. Split back into train and test
train_df = full_df[full_df['split'] == 'train'].copy()
test_df = full_df[full_df['split'] == 'test'].copy()

# Drop the 'split' column as it's no longer needed
train_df = train_df.drop(columns=['split'])
test_df = test_df.drop(columns=['split'])

# Display shapes to verify the split
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (77299, 18)
Test shape: (41778, 18)


In [8]:
# Temporal split within Day 48 to get a representative validation set
# Day 48 has all 96 timestamps (full 24 hours)
day48 = train_df[train_df['day'] == 48]
day49 = train_df[train_df['day'] == 49]

# Get sorted unique time_idx values for Day 48
day48_time_indices = sorted(day48['time_idx'].unique())
split_point = int(len(day48_time_indices) * 0.8)  # 80% mark
train_time_indices = day48_time_indices[:split_point]

# Training: first 80% of Day 48 (roughly hours 0:00 - 19:00)
train_fold = day48[day48['time_idx'].isin(train_time_indices)].copy()

# Validation: last 20% of Day 48 (hours ~19:15-23:45) + all of Day 49 (0:00-2:00)
val_from_day48 = day48[~day48['time_idx'].isin(train_time_indices)]
val_df = pd.concat([val_from_day48, day49], ignore_index=True)

print("Training fold shape:", train_fold.shape)
print("Validation fold shape:", val_df.shape)
print("Train hours covered:", sorted(train_fold['Hour'].unique()))
print("Val hours covered:", sorted(val_df['Hour'].unique()))


Training fold shape: (59585, 18)
Validation fold shape: (17714, 18)
Train hours covered: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Val hours covered: [0, 1, 2, 19, 20, 21, 22, 23]
